# RAG POC

## Stage 2: Ingest

Goal: turn raw MDX files into clean `Document`s, meaning text plus metadata.
Everything later depends on this stage. If the noise isn't removed here, it gets embedded and retrieved.

In [1]:
import re
from pathlib import Path
from langchain_core.documents import Document

DOCS = Path("data/langchain-docs/src/oss")

# ponytail: hand-picked subset (concepts + RAG/retrieval + core LangChain). Add pages later to see retrieval degrade.
SUBSET = [
    "langchain/overview", "langchain/philosophy", "langchain/quickstart", "langchain/component-architecture",
    "langchain/agents", "langchain/models", "langchain/messages", "langchain/tools",
    "langchain/structured-output", "langchain/short-term-memory", "langchain/long-term-memory",
    "langchain/streaming", "langchain/context-engineering", "langchain/guardrails",
    "langchain/human-in-the-loop", "langchain/knowledge-base", "langchain/retrieval",
    "langchain/middleware/overview", "langchain/middleware/built-in",
    "concepts/context", "concepts/memory", "concepts/providers-and-models", "concepts/products",
    "deepagents/retrieval", "deepagents/rag", "langgraph/overview", "langgraph/agentic-rag",
]

In [2]:
FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n", re.S)

def load(rel: str) -> Document:
    raw = (DOCS / f"{rel}.mdx").read_text()
    fm = FRONTMATTER.match(raw)
    title = re.search(r"^title:\s*(.+)$", fm.group(1), re.M) if fm else None
    body = raw[fm.end():] if fm else raw
    return Document(page_content=body, metadata={"source": rel, "title": title.group(1).strip('"') if title else rel})

raw_docs = [load(r) for r in SUBSET]
print(len(raw_docs), "docs |", sum(len(d.page_content) for d in raw_docs), "chars")

27 docs | 797759 chars


### Cleaning MDX

The noise found in these files:
- `import X from '/snippets/...'`: content pulled in from other files that we **don't** load
- `:::python ... :::` and `:::js ... :::`: the **same content written twice**, once per language
- JSX components: `<Note>`, `<Tip>`, `<CodeGroup>`, `<Tabs>`, `<ParamField path="...">`, ...
- `@[\`Document\`]`: the docs' own cross-reference syntax

In [3]:
def clean_mdx(text: str) -> str:
    text = re.sub(r"^[ \t]*:::js\n.*?^[ \t]*:::[ \t]*\n?", "", text, flags=re.S | re.M)        # drop JS duplicate content
    text = re.sub(r"^[ \t]*:::(?:python)?[ \t]*\n?", "", text, flags=re.M)             # unwrap Python blocks
    text = re.sub(r"^import .*$\n?", "", text, flags=re.M)                     # snippet imports (content not loaded)
    text = re.sub(r'<ParamField\s+[^>]*?path="([^"]+)"[^>]*>', r"Parameter `\1`:", text)  # keep param name
    text = re.sub(r"</?[A-Z][A-Za-z]*(?:\s[^>]*)?/?>", "", text)               # strip JSX tags, keep inner text
    text = re.sub(r"@\[(`?[^\]]+`?)\]", r"\1", text)                          # @[`Document`] -> `Document`
    return re.sub(r"\n{3,}", "\n\n", text).strip()                            # collapse blank lines left behind

In [4]:
docs = [Document(page_content=clean_mdx(d.page_content), metadata=d.metadata) for d in raw_docs]

before, after = sum(len(d.page_content) for d in raw_docs), sum(len(d.page_content) for d in docs)
print(f"chars: {before} -> {after} ({after/before:.0%} kept)")
for d in docs:
    if len(d.page_content.strip()) < 300:
        print("STUB?", d.metadata["source"], repr(d.page_content[:80]))

# Leftover noise check: should all be 0 or near it after cleaning
for name, pat in {"import": r"^import ", ":::js": r":::js", "JSX tag": r"</?[A-Z][A-Za-z]*", "@[ref]": r"@\["}.items():
    print(f"{name:8}", sum(len(re.findall(pat, d.page_content, re.M)) for d in docs))

chars: 797759 -> 502569 (63% kept)
STUB? langchain/retrieval ''
import   0
:::js    0
JSX tag  0
@[ref]   0


In [5]:
# Inspect: read a cleaned page the way the embedder will see it
print(docs[SUBSET.index("langchain/knowledge-base")].page_content[:2500])

## Overview

Build a semantic search engine over a PDF with LangChain [embeddings](/oss/integrations/embeddings) and [vector stores](/oss/integrations/vectorstores). Use it to retrieve passages similar to a query, then plug the retriever into [retrieval-augmented generation (RAG)](/oss/deepagents/retrieval) or other LLM workflows.

This tutorial covers:

1. Create `Document` objects from a PDF.
2. Generate embeddings.
3. Load and split a PDF.
4. Index chunks in a vector store and query by similarity.
5. Wrap the store as a retriever.

The guide also includes a minimal RAG implementation on top of the search engine.

### Concepts

This tutorial focuses on text retrieval and covers the following concepts:

- `Document`
- [Text splitters](/oss/integrations/splitters)
- [Embeddings](/oss/integrations/embeddings)
- [Vector stores](/oss/integrations/vectorstores) and [retrievers](/oss/integrations/retrievers)

## Setup

### Install dependencies

This tutorial reads a PDF using the `pypdf` pa

## Stage 3: Chunk

Why chunk at all: one vector per *page* averages many topics into one blurry point, so it matches everything a little and nothing well.
One vector per *chunk* is sharp, but a chunk cut in the wrong place loses its meaning (half a code block, a sentence without its heading).

Baseline first (naive), then measure what went wrong.

In [6]:
docs = [d for d in docs if len(d.page_content) >= 300]   # drop stub pages
print(len(docs), "docs")

26 docs


In [7]:
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
tok = AutoTokenizer.from_pretrained(os.environ["EMBED_MODEL"])  # the embedder counts tokens, not characters

def diagnose(chunks, name):
    lens = [len(c.page_content) for c in chunks]
    toks = [len(tok(c.page_content)["input_ids"]) for c in chunks]
    pct = lambda xs: [sorted(xs)[int(p * (len(xs) - 1))] for p in (0, .5, .95, 1)]
    cut_code = sum(c.page_content.count("```") % 2 for c in chunks)                   # odd ``` count = code block cut in half
    mid_start = sum(not re.match(r"[#A-Z`\-\d*\[|>]", c.page_content.lstrip()) for c in chunks)  # starts mid-sentence
    tiny = sum(l < 150 for l in lens)
    print(f"{name}: {len(chunks)} chunks | chars min/p50/p95/max {pct(lens)} | tokens {pct(toks)}")
    print(f"  cut code blocks: {cut_code} ({cut_code/len(chunks):.0%}) | mid-sentence starts: {mid_start} ({mid_start/len(chunks):.0%}) | tiny(<150): {tiny}")

In [8]:
baseline = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, add_start_index=True).split_documents(docs)
diagnose(baseline, "baseline")

baseline: 813 chunks | chars min/p50/p95/max [3, 679, 786, 800] | tokens [3, 162, 225, 343]
  cut code blocks: 270 (33%) | mid-sentence starts: 206 (25%) | tiny(<150): 14


In [9]:
# Inspect the failures: a chunk with half a code block, and a tiny one
bad_code = next(c for c in baseline if c.page_content.count("```") % 2)
tiny = min(baseline, key=lambda c: len(c.page_content))
for label, c in [("CUT CODE", bad_code), ("TINY", tiny)]:
    print(f"--- {label} | {c.metadata['source']} @ {c.metadata['start_index']}\n{c.page_content}\n")

--- CUT CODE | langchain/overview @ 1334
This example demonstrates how to create a simple LangChain agent with a custom tool:

    ```python OpenAI
    # pip install -qU langchain "langchain[openai]"
    from langchain.agents import create_agent

    def get_weather(city: str) -> str:
        """Get weather for a given city."""
        return f"It's always sunny in {city}!"

    agent = create_agent(
        model="openai:gpt-5.5",
        tools=[get_weather],
        system_prompt="You are a helpful assistant",
    )

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
    )
    print(result["messages"][-1].content_blocks)
    ```
    ```python Google Gemini
    # pip install -qU langchain "langchain[google-genai]"
    from langchain.agents import create_agent

--- TINY | langchain/streaming @ 11546
```



### Smarter separators

`RecursiveCharacterTextSplitter` tries separators **in order**: it splits on the first one, and only splits pieces that are still too big using the next one.
Default `["\n\n", "\n", " ", ""]` knows nothing about markdown. Your list decides where cuts are *preferred*.

In [10]:
from langchain_text_splitters import Language

# Headings first (any level), then fences, horizontal rules, paragraphs, lines, words.
# They are regexes, so is_separator_regex=True matters: without it they never match.
MD_SEPARATORS = RecursiveCharacterTextSplitter.get_separators_for_language(Language.MARKDOWN)
print(MD_SEPARATORS)

mine = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, separators=MD_SEPARATORS,
                                      is_separator_regex=True, add_start_index=True).split_documents(docs)
diagnose(baseline, "baseline")
diagnose(mine, "markdown")

['\n#{1,6} ', '```\n', '\n\\*\\*\\*+\n', '\n---+\n', '\n___+\n', '\n\n', '\n', ' ', '']
baseline: 813 chunks | chars min/p50/p95/max [3, 679, 786, 800] | tokens [3, 162, 225, 343]
  cut code blocks: 270 (33%) | mid-sentence starts: 206 (25%) | tiny(<150): 14
markdown: 1003 chunks | chars min/p50/p95/max [3, 553, 778, 798] | tokens [3, 133, 215, 335]
  cut code blocks: 346 (34%) | mid-sentence starts: 138 (14%) | tiny(<150): 90


## Stage 4: Embed + store

An embedding maps text to a point in 768-dimensional space, placed so that *meaning* determines the position: similar meaning, nearby points.
A vector store is just: keep every chunk's vector, then at query time compare the question's vector to all of them.

In [11]:
import time, numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

chunks = mine   # markdown-separator chunks from Stage 3

def make_embedder(prompts: bool) -> HuggingFaceEmbeddings:
    e = HuggingFaceEmbeddings(
        model_name=os.environ["EMBED_MODEL"],
        model_kwargs={"device": "cuda"},
        encode_kwargs={"prompt_name": "document", "batch_size": 32} if prompts else {"batch_size": 32},
        query_encode_kwargs={"prompt_name": "query"} if prompts else {},
    )
    e._client.max_seq_length = 8192   # model loads with this unset; cap it at the model card's limit
    return e

emb = make_embedder(prompts=True)
t = time.time()
store = InMemoryVectorStore.from_documents(chunks, emb)
print(f"embedded {len(chunks)} chunks in {time.time()-t:.1f}s")

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

embedded 1003 chunks in 6.6s


In [12]:
# What is a vector? Look at one.
V = np.array([rec["vector"] for rec in store.store.values()])
print("matrix:", V.shape, "| norms:", np.linalg.norm(V, axis=1).min().round(4), "-", np.linalg.norm(V, axis=1).max().round(4))
print("first 8 dims of chunk 0:", V[0][:8].round(3))
sims = V @ V.T                                         # normalized vectors, so dot product = cosine similarity
off = sims[~np.eye(len(V), dtype=bool)]
print(f"pairwise cosine across ALL chunk pairs: mean {off.mean():.3f}, p5 {np.percentile(off,5):.3f}, p95 {np.percentile(off,95):.3f}")

matrix: (1003, 768) | norms: 0.9958 - 1.0039
first 8 dims of chunk 0: [-0.018  0.048  0.03   0.012  0.037 -0.051  0.034 -0.053]
pairwise cosine across ALL chunk pairs: mean 0.670, p5 0.584, p95 0.757


In [13]:
# Near-duplicates: chunks that are almost the same point (e.g. the same example per provider tab)
recs = list(store.store.values())
i, j = np.where(np.triu(sims, 1) > 0.97)
print(f"{len(i)} chunk pairs with cosine > 0.97")
if len(i):
    a, b = recs[i[0]], recs[j[0]]
    print(f"cos={sims[i[0], j[0]]:.3f}\n--- {a['metadata']['source']}\n{a['text'][:300]}\n--- {b['metadata']['source']}\n{b['text'][:300]}")

708 chunk pairs with cosine > 0.97
cos=0.987
--- langchain/overview
```
    ```python Google Gemini
    # pip install -qU langchain "langchain[google-genai]"
    from langchain.agents import create_agent

    def get_weather(city: str) -> str:
        """Get weather for a given city."""
        return f"It's always sunny in {city}!"

    agent = create_agent(
      
--- langchain/quickstart
```
    ```python Google Gemini
    from langchain.agents import create_agent

    def get_weather(city: str) -> str:
        """Get weather for a given city."""
        return f"It's always sunny in {city}!"

    agent = create_agent(
        model="google_genai:gemini-2.5-flash-lite",
        tool


### Do the task prompts matter? A/B test

Probe questions with a known source page. Score = rank of the first chunk from that page (1 = best). EmbeddingGemma was trained with
`task: search result | query:` on questions and `title: none | text:` on documents, an **asymmetric** setup: short question vs long passage.

In [14]:
PROBES = {
    "How do I pause an agent so a human can approve a tool call before it runs?": "langchain/human-in-the-loop",
    "How can I make the agent return a Pydantic object instead of text?": "langchain/structured-output",
    "How do I get tokens as they are generated instead of waiting for the full reply?": "langchain/streaming",
    "How does the agent remember earlier messages in the same conversation?": "langchain/short-term-memory",
    "How do I build a semantic search over a PDF?": "langchain/knowledge-base",
    "What is agentic RAG?": "langgraph/agentic-rag",
    "How do I stop an agent from calling the model too many times?": "langchain/middleware/built-in",
    "What is the difference between LangChain and LangGraph?": "langchain/overview",
}

def first_hit_rank(store, q, source, k=50):
    for rank, (d, _) in enumerate(store.similarity_search_with_score(q, k=k), 1):
        if d.metadata["source"] == source:
            return rank
    return None   # not in top-k

# Same loaded model, different prompt settings (a second make_embedder() would load another 1.4 GB copy onto the GPU)
emb_raw = emb.model_copy(update={"encode_kwargs": {"batch_size": 32}, "query_encode_kwargs": {}})
store_raw = InMemoryVectorStore.from_documents(chunks, emb_raw)

print(f"{'with prompts':>12} {'no prompts':>11}  question")
for q, src in PROBES.items():
    print(f"{str(first_hit_rank(store, q, src)):>12} {str(first_hit_rank(store_raw, q, src)):>11}  {q[:60]}")

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

with prompts  no prompts  question
           1           1  How do I pause an agent so a human can approve a tool call b
           1           1  How can I make the agent return a Pydantic object instead of
           2           1  How do I get tokens as they are generated instead of waiting
           2           1  How does the agent remember earlier messages in the same con
           1           1  How do I build a semantic search over a PDF?
           3           3  What is agentic RAG?
           1           1  How do I stop an agent from calling the model too many times
          15          19  What is the difference between LangChain and LangGraph?


## Stage 5: Retrieve

Retrieval = embed the question with the *query* prompt, take the cosine with every chunk vector, return the top k.
No understanding, no keywords: only "which points sit closest to the question's point".

In [15]:
def show(q, k=15, store=store, width=110):
    hits = store.similarity_search_with_score(q, k=k)
    print(f"Q: {q}")
    for rank, (d, s) in enumerate(hits, 1):
        snippet = " ".join(d.page_content.split())[:width]
        print(f"{rank:>3} {s:.3f} {d.metadata['source']:<32} {len(d.page_content):>4}c  {snippet}")
    return hits

hits = show("What is the difference between LangChain and LangGraph?")

Q: What is the difference between LangChain and LangGraph?
  1 0.854 langgraph/overview                549c  We will commonly use [LangChain](/oss/langchain/overview) components throughout the documentation to integrate
  2 0.831 langgraph/overview                713c  Trusted by companies shaping the future of agents—including Klarna, Uber, J.P. Morgan, and more—LangGraph is a
  3 0.815 langchain/philosophy              717c  The main issue with building agents is getting them to be reliable, and LangSmith, which provides observabilit
  4 0.815 langgraph/overview                338c  ## Acknowledgements LangGraph is inspired by [Pregel](https://research.google/pubs/pub37252/) and [Apache Beam
  5 0.806 langchain/philosophy              522c  When building LangGraph, we learned from lessons when building LangChain and added functionality we discovered
  6 0.797 langgraph/overview                346c  One of LangGraph's core strengths is the ability to mix deterministic steps with LLM-d

In [16]:
# Where is the "real" answer? Every chunk that names both frameworks, and its rank for this question
q = "What is the difference between LangChain and LangGraph?"
all_hits = store.similarity_search_with_score(q, k=len(chunks))
both = [(r, s, d) for r, (d, s) in enumerate(all_hits, 1) if "LangChain" in d.page_content and "LangGraph" in d.page_content]
print(f"{len(both)} chunks mention both names. Their ranks:")
for r, s, d in both[:8]:
    print(f"{r:>4} {s:.3f} {d.metadata['source']:<28} {' '.join(d.page_content.split())[:150]}")

27 chunks mention both names. Their ranks:
   1 0.854 langgraph/overview           We will commonly use [LangChain](/oss/langchain/overview) components throughout the documentation to integrate models and tools, but you don't need to
   3 0.815 langchain/philosophy         The main issue with building agents is getting them to be reliable, and LangSmith, which provides observability and evals, was built to solve that nee
   4 0.815 langgraph/overview           ## Acknowledgements LangGraph is inspired by [Pregel](https://research.google/pubs/pub37252/) and [Apache Beam](https://beam.apache.org/). The public 
   5 0.806 langchain/philosophy         When building LangGraph, we learned from lessons when building LangChain and added functionality we discovered was needed: streaming, durable executio
   7 0.796 langgraph/overview           ## LangGraph ecosystem While LangGraph can be used standalone, it also integrates seamlessly with any LangChain product, giving developers a full suit
  

## Stage 6: Generate

The "G" in RAG: paste the retrieved chunks into the prompt and ask the LLM to answer **from them only**.
The LLM never sees the vector store. It only sees whatever text we hand it, so retrieval quality caps answer quality.

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=os.environ["LLM_MODEL"], base_url=os.environ["LLM_BASE_URL"],
                 api_key=os.environ["LLM_API_KEY"], temperature=0)   # temperature 0 = repeatable answers, easier to compare

SYSTEM = """You answer questions about LangChain using ONLY the context below.
Rules:
- Use only facts stated in the context. Do not use outside knowledge.
- If the context does not contain the answer, reply exactly: "Not in docs."
- Cite the source of every claim in square brackets, e.g. [langchain/overview].

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{question}")])
chain = prompt | llm | StrOutputParser()

def format_context(docs):
    return "\n\n---\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

def rag(q, k=4, store=store, show_prompt=False):
    docs = store.similarity_search(q, k=k)
    if show_prompt:
        print(prompt.format(context=format_context(docs), question=q))
    print(f"Q: {q}  (k={k})\nsources: {[d.metadata['source'] for d in docs]}\n")
    print(chain.invoke({"context": format_context(docs), "question": q}), "\n" + "=" * 100)

In [18]:
# The prediction: comparison question, k=4 (comparison table NOT retrieved)
rag("What is the difference between LangChain and LangGraph?", k=4)

Q: What is the difference between LangChain and LangGraph?  (k=4)
sources: ['langgraph/overview', 'langgraph/overview', 'langchain/philosophy', 'langgraph/overview']

LangChain provides LLM abstractions, high-level interfaces, and prebuilt architectures for common LLM and tool-calling loops [langgraph/overview, langchain/philosophy]. 

LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents [langgraph/overview]. It gives developers fine-grained control to mix deterministic steps with LLM steps and manage the exact flow of an agent, focusing on capabilities like durable execution, streaming, and human-in-the-loop [langgraph/overview, langchain/philosophy]. 

LangGraph is built by LangChain Inc, but works independently without requiring LangChain [langgraph/overview]. 


In [19]:
# Controls: same question with the table in reach (k=10), and a question the docs cannot answer
rag("What is the difference between LangChain and LangGraph?", k=10)
rag("How do I deploy a LangChain agent on Kubernetes with Helm?", k=4)

Q: What is the difference between LangChain and LangGraph?  (k=10)
sources: ['langgraph/overview', 'langgraph/overview', 'langchain/philosophy', 'langgraph/overview', 'langchain/philosophy', 'langgraph/overview', 'langgraph/overview', 'langchain/philosophy', 'concepts/products', 'concepts/products']

LangChain provides LLM abstractions, high-level interfaces, and prebuilt agent architectures for common applications and tool-calling loops [langchain/philosophy, langgraph/overview].

LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents [langgraph/overview]. It gives developers fine-grained control to mix deterministic steps with LLM-driven steps, focusing on capabilities like durable execution, streaming, and human-in-the-loop [langgraph/overview, langchain/philosophy]. 

Use LangChain for higher-level abstractions or when starting out; use LangGraph to control the exact orchestration flow of an application [lang

In [20]:
# See exactly what the LLM receives (the whole 'magic' of RAG is this string)
rag("How can I make the agent return a Pydantic object instead of text?", k=2, show_prompt=True)

System: You answer questions about LangChain using ONLY the context below.
Rules:
- Use only facts stated in the context. Do not use outside knowledge.
- If the context does not contain the answer, reply exactly: "Not in docs."
- Cite the source of every claim in square brackets, e.g. [langchain/overview].

Context:
[langchain/structured-output]
Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/#basic-model-usage), or dataclasses that your application can use directly.

This page covers structured output with agents using `create_agent`. To use structured output directly on a model (outside of agents), see [Models - Structured output](/oss/langchain/models#structured-output).

---

[langchain/tools]
#### Return a string

Return a string when the tool should provide plain text for the model 

## Stage 7: Evaluate

10 questions with answers taken from the docs. Each one tests a different way RAG can fail.
Two separate scores: **retrieval** (did the right page reach the top-4?) and **generation** (is the answer correct *and* grounded?).
Retrieval is scored automatically. Answers are scored by reading them against `expected`.

In [21]:
EVAL = [
    dict(type="easy lookup", q="What decisions can a human make when reviewing a tool call in human-in-the-loop?",
         expected="approve, edit, reject (also respond, for ask-user style tools)", sources=["langchain/human-in-the-loop"]),
    dict(type="paraphrase", q="My agent forgets what I told it two messages ago. How do I fix that?",
         expected="add a checkpointer (e.g. InMemorySaver) to create_agent and pass a thread_id in the config", sources=["langchain/short-term-memory"]),
    dict(type="answer in code", q="Which module do I import create_agent from?",
         expected="from langchain.agents import create_agent", sources=["langchain/agents", "langchain/overview", "langchain/quickstart"]),
    dict(type="exact default", q="What is the default exit_behavior of ModelCallLimitMiddleware, and what are the options?",
         expected="default 'end'; options 'end' (graceful) or 'error' (raise exception)", sources=["langchain/middleware/built-in"]),
    dict(type="number in code", q="In the SummarizationMiddleware example, at what token count does summarization trigger and how many messages are kept?",
         expected="trigger=('tokens', 4000), keep=('messages', 20)", sources=["langchain/middleware/built-in"]),
    dict(type="conditional rule", q="When does LangChain use ProviderStrategy instead of ToolStrategy for structured output?",
         expected="ProviderStrategy when the model/provider supports native structured output (e.g. OpenAI, Anthropic, xAI); ToolStrategy otherwise", sources=["langchain/structured-output"]),
    dict(type="specific value", q="Which stream_mode emits an event after every agent step?",
         expected='stream_mode="updates"', sources=["langchain/streaming"]),
    dict(type="ingestion gap", q="Which embedding model providers can I choose from in the semantic search tutorial?",
         expected="Not in docs (the provider tabs live in an imported /snippets file we never loaded)", sources=["langchain/knowledge-base"]),
    dict(type="partial corpus", q="What Python version do I need for LangChain?",
         expected="3.11+ per the indexed quickstart (the unindexed install page says 3.10+)", sources=["langchain/quickstart"]),
    dict(type="hallucination bait", q="What is the default chunk_size of RecursiveCharacterTextSplitter?",
         expected="Not in docs (the tutorial uses chunk_size=1000, but that is not the default)", sources=["langchain/knowledge-base"]),
]

In [22]:
def evaluate(store=store, k=4):
    rows = []
    for i, e in enumerate(EVAL, 1):
        docs = store.similarity_search(e["q"], k=k)
        got = [d.metadata["source"] for d in docs]
        rank = min((r for s in e["sources"] if (r := first_hit_rank(store, e["q"], s))), default=None)
        answer = chain.invoke({"context": format_context(docs), "question": e["q"]})
        rows.append(dict(i=i, hit=any(s in got for s in e["sources"]), rank=rank, got=got, answer=answer, **e))
    return rows

rows = evaluate()
print(f"retrieval hit@4: {sum(r['hit'] for r in rows)}/{len(rows)}\n")
for r in rows:
    print(f"Q{r['i']} [{r['type']}] hit@4={'Y' if r['hit'] else 'N'} best-rank={r['rank']}\n  Q: {r['q']}\n  expected: {r['expected']}\n  got from: {r['got']}\n  ANSWER: {r['answer']}\n")

retrieval hit@4: 10/10

Q1 [easy lookup] hit@4=Y best-rank=1
  Q: What decisions can a human make when reviewing a tool call in human-in-the-loop?
  expected: approve, edit, reject (also respond, for ask-user style tools)
  got from: ['langchain/human-in-the-loop', 'langchain/human-in-the-loop', 'langchain/human-in-the-loop', 'langchain/human-in-the-loop']
  ANSWER: Four decision types exist for tool call reviews [langchain/human-in-the-loop]:

- `approve`: Approve the action as-is [langchain/human-in-the-loop].
- `edit`: Modify the action before running [langchain/human-in-the-loop].
- `reject`: Reject the action with feedback [langchain/human-in-the-loop].
- `respond`: Respond directly for "ask user" style tools [langchain/human-in-the-loop].

Q2 [paraphrase] hit@4=Y best-rank=1
  Q: My agent forgets what I told it two messages ago. How do I fix that?
  expected: add a checkpointer (e.g. InMemorySaver) to create_agent and pass a thread_id in the config
  got from: ['langchain/short-t

In [23]:
# Prompt A/B again, now on 10 questions instead of 8 probes (rank of best expected page; lower = better)
for e in EVAL:
    a = min((r for s in e["sources"] if (r := first_hit_rank(store, e["q"], s))), default=None)
    b = min((r for s in e["sources"] if (r := first_hit_rank(store_raw, e["q"], s))), default=None)
    print(f"{str(a):>5} {str(b):>5}  {e['type']:<18} {e['q'][:60]}")

    1     1  easy lookup        What decisions can a human make when reviewing a tool call i
    1     4  paraphrase         My agent forgets what I told it two messages ago. How do I f
    1     1  answer in code     Which module do I import create_agent from?
    1     1  exact default      What is the default exit_behavior of ModelCallLimitMiddlewar
    1     1  number in code     In the SummarizationMiddleware example, at what token count 
    1     1  conditional rule   When does LangChain use ProviderStrategy instead of ToolStra
    1     1  specific value     Which stream_mode emits an event after every agent step?
    1     1  ingestion gap      Which embedding model providers can I choose from in the sem
    1     1  partial corpus     What Python version do I need for LangChain?
    1     2  hallucination bait What is the default chunk_size of RecursiveCharacterTextSpli


## Stage 8: Improve

Change **one thing at a time** and re-run the same 10 questions. Otherwise you can't tell which change helped.

| Version | Cleaner | Chunking |
|---|---|---|
| v1 (Stage 7) | original | markdown separators |
| v2 | fixed `ParamField` | markdown separators |
| v3 | fixed `ParamField` | header-aware + heading path prefixed to each chunk |

### Automatic grading: LLM-as-judge
Reading 10 answers by hand per version doesn't scale. A second LLM call compares each answer to `expected`.
The judge can be wrong too, so spot-check its verdicts.

In [24]:
JUDGE = ChatPromptTemplate.from_messages([("human", """Grade an answer against the expected answer.
Question: {q}
Expected: {expected}
Answer: {answer}

CORRECT if the answer contains the expected facts (extra correct detail is fine; if expected says "Not in docs", the answer must decline).
WRONG if it misses a key fact, contradicts the expected answer, or answers when it should decline.
Reply with one word: CORRECT or WRONG.""")])
judge = JUDGE | llm | StrOutputParser()

def grade(rows):
    for r in rows:
        r["ok"] = judge.invoke(r).strip().upper().startswith("CORRECT")
    return rows

def report(rows, name):
    marks = " ".join(f"Q{r['i']}{'✓' if r['ok'] else '✗'}" for r in rows)
    print(f"{name:<4} retrieval {sum(r['hit'] for r in rows)}/10 | answers {sum(r['ok'] for r in rows)}/10 | {marks}")

v1 = grade(rows)   # Stage 7 results, graded by the judge
report(v1, "v1")

v1   retrieval 10/10 | answers 8/10 | Q1✓ Q2✗ Q3✓ Q4✗ Q5✓ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓


### v2: fix the cleaner
`<ParamField body="exit_behavior" default="end">` lost both the name and the default. Turn the tag into text instead of deleting it.

In [25]:
def paramfield_to_text(m):
    attrs = dict(re.findall(r'(\w+)="([^"]*)"', m.group(0)))
    name = attrs.get("path") or attrs.get("body") or attrs.get("query") or "?"
    extra = ", ".join(f"{k}: {attrs[k]}" for k in ("type", "default") if k in attrs)
    return f"Parameter `{name}`" + (f" ({extra})" if extra else "") + ":"

def clean_mdx_v2(text):
    text = re.sub(r"<ParamField\s[^>]*>", paramfield_to_text, text)   # must run before clean_mdx strips all tags
    return clean_mdx(text)

docs_v2 = [Document(page_content=clean_mdx_v2(d.page_content), metadata=d.metadata) for d in raw_docs]
docs_v2 = [d for d in docs_v2 if len(d.page_content) >= 300]
print(sum(d.page_content.count("Parameter `") for d in docs_v2), "parameters kept (v1 kept",
      sum(d.page_content.count("Parameter `") for d in docs), ")")
print([l for d in docs_v2 for l in d.page_content.splitlines() if "exit_behavior" in l and "Parameter" in l][:2])

def split_md(ds):
    return RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, separators=MD_SEPARATORS,
                                          is_separator_regex=True, add_start_index=True).split_documents(ds)

chunks_v2 = split_md(docs_v2)
store_v2 = InMemoryVectorStore.from_documents(chunks_v2, emb)
v2 = grade(evaluate(store_v2)); report(v1, "v1"); report(v2, "v2")

162 parameters kept (v1 kept 16 )
['Parameter `exit_behavior` (type: string, default: end):', 'Parameter `exit_behavior` (type: string, default: continue):']
v1   retrieval 10/10 | answers 8/10 | Q1✓ Q2✗ Q3✓ Q4✗ Q5✓ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓
v2   retrieval 10/10 | answers 9/10 | Q1✓ Q2✗ Q3✓ Q4✓ Q5✓ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓


### v3: header-aware chunks with the heading path in front
Split on headings first, so each chunk belongs to exactly one section, then split long sections by size.
Prefix every chunk with `Page title > Section > Subsection`. A code-only chunk then still says *what* it is about,
and that prefix gets embedded with the chunk.

In [26]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

def split_headers(ds):
    by_header = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2"), ("###", "h3")], strip_headers=True)
    sized = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, separators=MD_SEPARATORS, is_separator_regex=True)
    out = []
    for d in ds:
        for sec in by_header.split_text(d.page_content):          # knows ``` fences: a "#" comment in code is not a heading
            path = " > ".join([d.metadata["title"]] + [sec.metadata[h] for h in ("h1", "h2", "h3") if h in sec.metadata])
            for piece in sized.split_text(sec.page_content):
                out.append(Document(page_content=f"{path}\n\n{piece}", metadata={**d.metadata, "section": path}))
    return out

chunks_v3 = split_headers(docs_v2)
diagnose(chunks_v2, "v2 chunks")
diagnose(chunks_v3, "v3 chunks")
print("\nexample v3 chunk:\n", next(c for c in chunks_v3 if "checkpointer" in c.page_content).page_content[:500])

store_v3 = InMemoryVectorStore.from_documents(chunks_v3, emb)
v3 = grade(evaluate(store_v3)); report(v1, "v1"); report(v2, "v2"); report(v3, "v3")

v2 chunks: 1011 chunks | chars min/p50/p95/max [3, 556, 778, 798] | tokens [3, 135, 215, 335]
  cut code blocks: 348 (34%) | mid-sentence starts: 137 (14%) | tiny(<150): 90
v3 chunks: 923 chunks | chars min/p50/p95/max [38, 598, 840, 901] | tokens [9, 149, 241, 350]
  cut code blocks: 317 (34%) | mid-sentence starts: 0 (0%) | tiny(<150): 27

example v3 chunk:
 Quickstart > Build a real-world agent

checkpointer = InMemorySaver()
```  
In production, use a persistent checkpointer that saves message history to a database.
See [Add and manage memory](/oss/langgraph/add-memory#manage-short-term-memory) for more details.  
Now assemble your agent with all the components and run it.  
There are two different frameworks for creating agents: LangChain agents and deep agents.
Both LangChain and deep agents provide you with fine-grained control over tools, memo
v1   retrieval 10/10 | answers 8/10 | Q1✓ Q2✗ Q3✓ Q4✗ Q5✓ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓
v2   retrieval 10/10 | answers 9/10 | Q1✓ Q2✗ Q3✓ Q4✓ Q5✓

In [27]:
# Look at what changed for the two Stage 7 failures
for r in v3:
    if r["i"] in (2, 4):
        print(f"Q{r['i']}: {r['q']}\n  sources: {r['got']}\n  ANSWER: {r['answer']}\n")

Q2: My agent forgets what I told it two messages ago. How do I fix that?
  sources: ['langchain/short-term-memory', 'langchain/short-term-memory', 'langchain/short-term-memory', 'langchain/context-engineering']
  ANSWER: Not in docs.

Q4: What is the default exit_behavior of ModelCallLimitMiddleware, and what are the options?
  sources: ['langchain/middleware/built-in', 'langchain/middleware/built-in', 'langchain/middleware/built-in', 'langchain/middleware/built-in']
  ANSWER: The default `exit_behavior` of `ModelCallLimitMiddleware` is `'end'` [langchain/middleware/built-in].

Options:
- `'end'`: Graceful termination [langchain/middleware/built-in].
- `'error'`: Raise exception [langchain/middleware/built-in].



### How many chunks to retrieve? k = 1 / 4 / 15 on v3

In [28]:
for k in (1, 4, 15):
    report(grade(evaluate(store_v3, k=k)), f"k={k}")

k=1  retrieval 10/10 | answers 6/10 | Q1✓ Q2✗ Q3✗ Q4✓ Q5✗ Q6✓ Q7✗ Q8✓ Q9✓ Q10✓
k=4  retrieval 10/10 | answers 8/10 | Q1✓ Q2✗ Q3✓ Q4✓ Q5✗ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓
k=15 retrieval 10/10 | answers 10/10 | Q1✓ Q2✓ Q3✓ Q4✓ Q5✓ Q6✓ Q7✓ Q8✓ Q9✓ Q10✓


### Provoking a made-up answer
Same retrieval (v3, k=4). Remove the grounding rules and raise the temperature. Does the model start answering from its own knowledge?

In [29]:
loose = ChatPromptTemplate.from_messages([("system", "You are a helpful LangChain expert. Here is some documentation that may help:\n\n{context}"),
                                          ("human", "{question}")]) | llm.bind(temperature=1.0) | StrOutputParser()
for i in (8, 10):
    e = EVAL[i - 1]
    ctx = format_context(store_v3.similarity_search(e["q"], k=4))
    print(f"Q{i}: {e['q']}\n  expected: {e['expected']}\n  LOOSE ANSWER: {loose.invoke({'context': ctx, 'question': e['q']})[:700]}\n")

Q8: Which embedding model providers can I choose from in the semantic search tutorial?
  expected: Not in docs (the provider tabs live in an imported /snippets file we never loaded)
  LOOSE ANSWER: The documentation excerpt links broadly to `/oss/integrations/embeddings/` without listing specific options inline. 

LangChain's standard tutorial allows choosing from these common providers:
- **OpenAI** (`langchain-openai`)
- **Cohere** (`langchain-cohere`)
- **Hugging Face** (`langchain-huggingface`)
- **Google GenAI** (`langchain-google-genai`)
- **AWS Bedrock** (`langchain-aws`)
- **Ollama** (`langchain-ollama`) (local)

Select one based on whether you want a cloud API or a local model. Check `/oss/integrations/embeddings/` for the complete list.

Q10: What is the default chunk_size of RecursiveCharacterTextSplitter?
  expected: Not in docs (the tutorial uses chunk_size=1000, but that is not the default)
  LOOSE ANSWER: The default `chunk_size` is `4000`. 

Defined in the `TextSplitter

### Persist: move v3 to Chroma
`InMemoryVectorStore` is lost on every restart, so every run re-embeds everything. Chroma writes to disk; the next run only loads.
Chroma's default distance is **L2**, not cosine. Ask for cosine explicitly so the scores mean the same thing as before.

In [30]:
from langchain_chroma import Chroma

CHROMA_DIR = "data/chroma"
chroma = Chroma(collection_name="langchain-docs-v3", embedding_function=emb, persist_directory=CHROMA_DIR,
                collection_metadata={"hnsw:space": "cosine"})
if chroma._collection.count() == 0:            # ponytail: naive "empty = build"; a changed corpus needs a rebuild (delete data/chroma)
    chroma.add_documents(chunks_v3)
print("chroma chunks:", chroma._collection.count())

for (a, sa), (b, sb) in zip(store_v3.similarity_search_with_score(EVAL[1]["q"], k=4),
                            chroma.similarity_search_with_score(EVAL[1]["q"], k=4)):
    print(f"memory {sa:.3f} {a.metadata['section'][:45]:<45} | chroma {sb:.3f} {b.metadata['section'][:45]}")

chroma chunks: 923
memory 0.772 Short-term memory > Common patterns > Delete  | chroma 0.228 Short-term memory > Common patterns > Delete 
memory 0.747 Short-term memory > Customizing agent memory  | chroma 0.253 Short-term memory > Customizing agent memory
memory 0.740 Short-term memory > Access memory > Tools     | chroma 0.260 Short-term memory > Access memory > Tools
memory 0.735 Context engineering in agents > Model context | chroma 0.265 Context engineering in agents > Model context
